# Classic ML Baselines

This notebook runs the classic baseline grid and creates two analysis tables:

- `total_results`: all evaluated combinations.
- `report_results`: the best validation result per representation family.

In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd

EXPERIMENT_DIR = Path("experiments/classic_ml")
EMBEDDING_DIR = Path("experiments/embeddings")
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
EMBEDDING_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = Path("data/train.csv")
VALIDATION_SIZE = 0.1
RANDOM_STATE = 42
MAX_FEATURES = 200_000
MAX_ITER = 100

embedding_paths = {
    "glove": EMBEDDING_DIR / "glove.6B.300d.txt",
    "fasttext": EMBEDDING_DIR / "fasttext.vec",
}

available_embeddings = {
    name: path for name, path in embedding_paths.items() if path.exists()
}
available_embeddings

Place optional pretrained static embeddings in `experiments/embeddings/` with the filenames configured above. If no embedding files are present, the notebook runs the sparse baselines only.

In [ ]:
mode = "all" if available_embeddings else "sparse"

cmd = [
    sys.executable,
    "-m",
    "baselines.classic_ml_baselines",
    "--mode",
    mode,
    "--train-path",
    str(TRAIN_PATH),
    "--output-dir",
    str(EXPERIMENT_DIR),
    "--validation-size",
    str(VALIDATION_SIZE),
    "--random-state",
    str(RANDOM_STATE),
    "--max-features",
    str(MAX_FEATURES),
    "--max-iter",
    str(MAX_ITER),
]

for name, path in available_embeddings.items():
    cmd.extend([f"--{name}-path", str(path)])

print(" ".join(cmd))
subprocess.run(cmd, check=True)

## Total Analysis

In [ ]:
results = pd.read_csv(EXPERIMENT_DIR / "classic_ml_results.csv")
metric_columns = [
    "cil_score",
    "mae",
    "accuracy",
    "macro_f1",
    "quadratic_weighted_kappa",
]

total_results = results.sort_values(
    ["status", "cil_score"], ascending=[False, False]
).reset_index(drop=True)
total_results.to_csv(EXPERIMENT_DIR / "total_analysis.csv", index=False)
total_results

## Report Analysis

In [ ]:
ok_results = results[results["status"] == "ok"].copy()
report_results = (
    ok_results.sort_values("cil_score", ascending=False)
    .groupby("family", as_index=False)
    .first()
)

report_columns = [
    "family",
    "representation",
    "variant",
    "classifier",
    "cil_score",
    "mae",
    "accuracy",
    "macro_f1",
    "quadratic_weighted_kappa",
    "train_seconds",
]
report_results = report_results[report_columns].sort_values(
    "cil_score", ascending=False
).reset_index(drop=True)
report_results.to_csv(EXPERIMENT_DIR / "report_analysis.csv", index=False)
report_results